# Marathi ASR Training on Google Colab

## 📋 Setup Instructions

### Before Running:
1. **Enable GPU**: Runtime → Change runtime type → GPU (T4 or better)
2. **Upload your data**: Use the file browser or mount Google Drive
3. **Run cells in order**

### Expected Results:
- Training time: 2-3 hours for 50 epochs
- Target WER: <12%
- Free GPU time: ~12 hours on Colab

---

## Cell 1: Mount Google Drive (Optional)

In [ ]:
# Cell 1: Mount Google Drive to access your data
from google.colab import drive
import os

drive.mount('/content/drive')

# Set your data directory (adjust this path!)
# Option 1: If you uploaded to Drive
DATA_DIR = '/content/drive/MyDrive/marathi-asr-data'

# Option 2: If you'll upload directly to Colab
# DATA_DIR = '/content/marathi-asr-data'

print(f"Data directory: {DATA_DIR}")
print(f"Exists: {os.path.exists(DATA_DIR)}")

## Cell 2: Upload Data (Alternative to Drive)

In [ ]:
# Cell 2: Upload your training data ZIP file
# Skip this if you're using Google Drive

from google.colab import files
import zipfile

print("Upload your kaggle_training_complete.zip file:")
uploaded = files.upload()

# Extract
zip_file = list(uploaded.keys())[0]
print(f"\nExtracting {zip_file}...")
with zipfile.ZipFile(zip_file, 'r') as zip_ref:
    zip_ref.extractall('/content/')

DATA_DIR = '/content/kaggle_training_complete'
print(f"✓ Extracted to {DATA_DIR}")

## Cell 3: Install Dependencies

In [ ]:
# Cell 3: Install required packages
!pip install -q librosa soundfile sentencepiece pyyaml jiwer

print("✓ Dependencies installed!")

## Cell 4: Setup Python Path

In [ ]:
# Cell 4: Add code to Python path
import sys
import os

# Add marathi_asr to path
sys.path.insert(0, os.path.join(DATA_DIR, 'marathi_asr'))
sys.path.insert(0, DATA_DIR)

print(f"✓ Added {DATA_DIR} to Python path")

# Verify structure
print("\nChecking directory structure:")
for item in ['marathi_asr', 'kaggle_model_output', 'synthetic_audio']:
    path = os.path.join(DATA_DIR, item)
    exists = "✓" if os.path.exists(path) else "✗"
    print(f"  {exists} {item}")

## Cell 5: Verify GPU

In [ ]:
# Cell 5: Check GPU availability
import torch

print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("⚠️ NO GPU! Go to Runtime → Change runtime type → GPU")

print("\n✓ Setup complete!")

## Cell 6: Import Modules

In [ ]:
# Cell 6: Import all required modules
from marathi_asr.config import ASRConfig
from marathi_asr.models import MarathiASRModel
from marathi_asr.models.encoder import TransformerEncoder
from marathi_asr.models.decoder import TransformerDecoder
from marathi_asr.data import MarathiASRDataset
from marathi_asr.data.tokenizer import MarathiTokenizer
from marathi_asr.data.feature_extractor import FeatureExtractor
from marathi_asr.training import Trainer
from torch.utils.data import DataLoader
import time

print("✓ All modules imported!")

## Cell 7: Configuration

In [ ]:
# Cell 7: Training configuration
config = ASRConfig(
    encoder_layers=6,
    decoder_layers=4,
    model_dim=384,
    num_heads=6,
    ff_dim=1536,
    dropout=0.1,
    sample_rate=16000,
    n_mels=80,
    win_length_ms=25.0,
    hop_length_ms=10.0,
    normalize_features=True,
    vocab_size=1000,
    batch_size=32,
    learning_rate=0.0001,
    num_epochs=50,
    checkpoint_interval=5,
    use_fp16=True,
    device="cuda" if torch.cuda.is_available() else "cpu"
)

print("✓ Configuration loaded")
print(f"  Epochs: {config.num_epochs}")
print(f"  Batch size: {config.batch_size}")
print(f"  Device: {config.device}")

## Cell 8: Load Tokenizer

In [ ]:
# Cell 8: Load tokenizer
print("Loading tokenizer...")
tokenizer = MarathiTokenizer(vocab_size=config.vocab_size)

# Try to find tokenizer
tokenizer_paths = [
    os.path.join(DATA_DIR, 'marathi_tokenizer'),
    os.path.join(DATA_DIR, 'kaggle_model_output', 'marathi_tokenizer'),
    '/content/marathi_tokenizer'
]

tokenizer_loaded = False
for path in tokenizer_paths:
    if os.path.exists(path + '.model'):
        tokenizer.load(path)
        print(f"✓ Tokenizer loaded from {path}")
        tokenizer_loaded = True
        break

if not tokenizer_loaded:
    print("⚠️ Tokenizer not found. Please upload marathi_tokenizer.model and .vocab")
    raise FileNotFoundError("Tokenizer files missing")

print(f"✓ Tokenizer ready (vocab: {tokenizer.vocab_size})")

## Cell 9: Create Model and Load Checkpoint

In [ ]:
# Cell 9: Create model
print("Creating model...")

encoder = TransformerEncoder(
    input_dim=config.n_mels,
    model_dim=config.model_dim,
    num_layers=config.encoder_layers,
    num_heads=config.num_heads,
    ff_dim=config.ff_dim,
    dropout=config.dropout
)

decoder = TransformerDecoder(
    vocab_size=tokenizer.vocab_size,
    model_dim=config.model_dim,
    num_layers=config.decoder_layers,
    num_heads=config.num_heads,
    ff_dim=config.ff_dim,
    dropout=config.dropout
)

model = MarathiASRModel(encoder=encoder, decoder=decoder, tokenizer=tokenizer)

# Load checkpoint if available
checkpoint_paths = [
    os.path.join(DATA_DIR, 'kaggle_model_output', 'best_model_wer.pt'),
    os.path.join(DATA_DIR, 'best_model_wer.pt'),
    '/content/best_model_wer.pt'
]

checkpoint = None
for path in checkpoint_paths:
    if os.path.exists(path):
        checkpoint = torch.load(path, map_location='cpu')
        model.load_state_dict(checkpoint['model_state_dict'])
        print(f"✓ Loaded checkpoint from {path}")
        break

if checkpoint:
    current_epoch = checkpoint.get('epoch', 0)
    best_wer = checkpoint.get('best_wer', float('inf'))
    best_loss = checkpoint.get('best_loss', float('inf'))
    print(f"  Starting from epoch {current_epoch}")
    print(f"  Current WER: {best_wer:.2f}%")
else:
    print("⚠️ No checkpoint found, starting from scratch")
    current_epoch = 0
    best_wer = float('inf')
    best_loss = float('inf')

model = model.to(config.device)
print(f"\n✓ Model ready on {config.device}")

## Cell 10: Load Datasets

In [ ]:
# Cell 10: Load training and validation datasets
print("Loading datasets...")

feature_extractor = FeatureExtractor(
    sample_rate=config.sample_rate,
    n_mels=config.n_mels,
    win_length_ms=config.win_length_ms,
    hop_length_ms=config.hop_length_ms,
    normalize=config.normalize_features
)

# Find manifest files
train_manifest = os.path.join(DATA_DIR, 'synthetic_audio', 'train_manifest.csv')
val_manifest = os.path.join(DATA_DIR, 'synthetic_audio', 'val_manifest.csv')

if not os.path.exists(train_manifest) or not os.path.exists(val_manifest):
    raise FileNotFoundError(f"Manifest files not found in {DATA_DIR}/synthetic_audio/")

print(f"✓ Train manifest: {train_manifest}")
print(f"✓ Val manifest: {val_manifest}")

# Create datasets
train_dataset = MarathiASRDataset(
    manifest_path=train_manifest,
    audio_dir=DATA_DIR,
    feature_extractor=feature_extractor,
    tokenizer=tokenizer,
    augment=True
)

val_dataset = MarathiASRDataset(
    manifest_path=val_manifest,
    audio_dir=DATA_DIR,
    feature_extractor=feature_extractor,
    tokenizer=tokenizer,
    augment=False
)

print(f"✓ Train: {len(train_dataset)} samples")
print(f"✓ Val: {len(val_dataset)} samples")

## Cell 11: Create Data Loaders

In [ ]:
# Cell 11: Create data loaders
def collate_fn(batch):
    features_list, tokens_list = [], []
    for features, tokens in batch:
        features_list.append(features)
        tokens_list.append(tokens)
    
    # Pad features
    max_feat_len = max(f.shape[1] for f in features_list)
    padded_features = []
    for f in features_list:
        if f.shape[1] < max_feat_len:
            f = torch.nn.functional.pad(f, (0, max_feat_len - f.shape[1]))
        padded_features.append(f)
    features_batch = torch.stack(padded_features)
    
    # Pad tokens
    max_token_len = max(len(t) for t in tokens_list)
    padded_tokens = []
    for t in tokens_list:
        if len(t) < max_token_len:
            t = torch.nn.functional.pad(t, (0, max_token_len - len(t)), value=tokenizer.pad_token_id)
        padded_tokens.append(t)
    tokens_batch = torch.stack(padded_tokens)
    
    return features_batch, tokens_batch

# Create loaders (num_workers=0 for Colab stability)
train_loader = DataLoader(
    train_dataset, 
    batch_size=config.batch_size, 
    shuffle=True, 
    num_workers=0,  # 0 for Colab
    pin_memory=True, 
    collate_fn=collate_fn
)

val_loader = DataLoader(
    val_dataset, 
    batch_size=config.batch_size, 
    shuffle=False, 
    num_workers=0,  # 0 for Colab
    pin_memory=True, 
    collate_fn=collate_fn
)

print(f"✓ Train batches: {len(train_loader)}")
print(f"✓ Val batches: {len(val_loader)}")

## Cell 12: Setup Trainer

In [ ]:
# Cell 12: Setup trainer and optimizer
optimizer = torch.optim.AdamW(model.parameters(), lr=config.learning_rate, weight_decay=0.01)

if checkpoint and 'optimizer_state_dict' in checkpoint:
    optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
    print("✓ Optimizer state restored")

trainer = Trainer(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    optimizer=optimizer,
    config=config,
    device=config.device
)

print("✓ Trainer ready!")
print(f"\nStarting training from epoch {current_epoch + 1}")
if best_wer < float('inf'):
    print(f"Target: Reduce WER from {best_wer:.2f}% to <12%")

## Cell 13: START TRAINING!

⚠️ **This will take 2-3 hours**

Expected progress:
- ~3-4 minutes per epoch
- WER should decrease gradually
- Best model saved automatically to `/content/`

**Important:** Colab may disconnect after 12 hours. Download checkpoints periodically!

In [ ]:
# Cell 13: TRAINING LOOP
print("=" * 80)
print(f"TRAINING FOR {config.num_epochs} EPOCHS")
print("=" * 80)

training_start = time.time()
history = {'train_loss': [], 'val_loss': [], 'val_wer': [], 'val_cer': []}

for epoch in range(config.num_epochs):
    actual_epoch = current_epoch + epoch + 1
    epoch_start = time.time()
    
    print(f"\nEpoch {actual_epoch}/{current_epoch + config.num_epochs}")
    print("-" * 80)
    
    train_loss = trainer.train_epoch()
    val_metrics = trainer.validate()
    
    epoch_time = time.time() - epoch_start
    total_time = time.time() - training_start
    
    history['train_loss'].append(train_loss)
    history['val_loss'].append(val_metrics['loss'])
    history['val_wer'].append(val_metrics['wer'])
    history['val_cer'].append(val_metrics['cer'])
    
    print(f"Train Loss: {train_loss:.4f}")
    print(f"Val Loss: {val_metrics['loss']:.4f}")
    print(f"Val WER: {val_metrics['wer']:.2f}%")
    print(f"Val CER: {val_metrics['cer']:.2f}%")
    print(f"Time: {epoch_time/60:.1f} min | Total: {total_time/3600:.2f} hrs")
    
    # Save checkpoint every 10 epochs
    if (epoch + 1) % 10 == 0:
        checkpoint_path = f"/content/checkpoint_epoch_{actual_epoch}.pt"
        torch.save({
            'epoch': actual_epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'best_wer': best_wer,
            'best_loss': best_loss
        }, checkpoint_path)
        print(f"✓ Checkpoint saved: {checkpoint_path}")
    
    # Save best models
    if val_metrics['wer'] < best_wer:
        best_wer = val_metrics['wer']
        torch.save({
            'epoch': actual_epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'best_wer': best_wer,
            'best_loss': best_loss
        }, "/content/best_model_wer.pt")
        print(f"✓ NEW BEST WER: {best_wer:.2f}%")
    
    if val_metrics['loss'] < best_loss:
        best_loss = val_metrics['loss']
        torch.save({
            'epoch': actual_epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'best_wer': best_wer,
            'best_loss': best_loss
        }, "/content/best_model_loss.pt")

print("\n" + "=" * 80)
print("✅ TRAINING COMPLETE!")
print("=" * 80)
print(f"Final WER: {best_wer:.2f}%")
print(f"Total time: {(time.time() - training_start)/3600:.2f} hours")
print(f"\nModels saved to /content/")
print("Download them before closing Colab!")

## Cell 14: Download Trained Model

In [ ]:
# Cell 14: Download the best model
from google.colab import files

print("Downloading best model...")
files.download('/content/best_model_wer.pt')
print("✓ Downloaded!")

---

## 🎉 Training Complete!

### Next Steps:

1. **Download your model**: Already done in Cell 14!
2. **Test locally**: Use `test_trained_model.py`
3. **Run demo**: Use `phoneme_asr_demo.py`

### Tips:

- **Save to Drive**: Copy models to Google Drive for safekeeping
- **Monitor GPU**: Check Runtime → Manage sessions
- **Resume training**: Upload checkpoint and adjust `current_epoch`

---